# OCR · 번역 실험 (text/extract_and_translate 분해)

**제품 비교 대상**은 `text/extract_and_translate` 통합 동작이다. 이미지 한 장을 한 번 불러 원문(`original`)과 번역(`translation`)을 같이 받는다. 이 notebook은 그것을 나눠서 본다.

| run | 입력 | 출력 | Go 채점 |
| --- | --- | --- | --- |
| OCR | 이미지 | 읽은 원문 | text-extraction replay |
| 번역(gold) | 확정 원문 · 출발 · 목표 언어 | 번역 | translation replay |
| 연쇄 진단 | OCR 후보가 **실제로 낸** 원문 | 번역 | 없음. 연구 기록만(`input: ocr-output`). gold 번역 점수와 합치지 않는다 |
| 같은 이미지 기준선 | 이미지 | 원문 + 번역 | 원문 부분만 text-extraction replay. 번역 부분은 E2E 연구 기록 |

기준선:

- 같은 이미지 기준선 — 제품 통합 지시문(`prompts/product-extract-and-translate.md`) · 운영 모델. **실제 기존 경로 출력은 아직 없다 → 미측정**
- 독립 과제 기준선 — OCR은 제품 extract_text 지시문, 번역은 `prompts/translate-text-v1.md`. 공급자 호출 전까지 **미측정**. 규칙 바닥값(빈 원문 · 원문 복사)은 모델 성능이 아니다

`MODE`:

- `"fixture"`(기본) — `fixtures/smoke`의 고정 답을 재생. **smoke 자료**이고 품질 근거가 아니다
- `"dry-run"` — 아무것도 부르지 않고 모든 case가 not-run
- `"provider"` — 실제 호출. `ALLOW_EXTERNAL = True`(공급자 · 데이터 외부 전송 승인)와 `MAX_CALLS`(case 수 상한. anthropic SDK 재시도는 그 안에서 최대 2번 더)가 필요하고, case마다 `transfer:external-ok` 태그가 있어야 한다

dev에서만 후보를 고른다. validation은 고른 뒤 점검, held-out은 열지 않는다.

In [ ]:
from pathlib import Path

from snapdone_eval_lab import tasks as t
from snapdone_eval_lab.datasets import load_dataset, repository_root
from snapdone_eval_lab.experiments import load_prompt
from snapdone_eval_lab.research import RunPlan, records_frame, transfer_blockers, unique_run_id

ROOT = repository_root()
LAB = ROOT / "tools/evals/lab"
OUT = LAB / "out"
SMOKE = LAB / "fixtures/smoke"
VARIANTS = ROOT / "tools/evals/variants"

# ---- 실행 설정 -------------------------------------------------------------
MODE = "fixture"                  # "fixture" · "dry-run" · "provider"
ALLOW_EXTERNAL = False            # provider일 때만: 공급자 · 데이터 외부 전송 승인
PROVIDER = "anthropic"            # "anthropic" 또는 "openai"(OpenAI 호환: Ollama · vLLM)
MODEL = "claude-haiku-5-5"        # 운영 설정(apps/api/service.yaml)의 Git 값
BASE_URL = None
MAX_CALLS = 10                    # run 하나의 공급자 호출(case) 수 상한
SPLIT = "dev"
BASE_ID = "lab-text"

PLAN = RunPlan(
    hypothesis="통합 동작을 OCR과 번역으로 나누면 오류가 어느 단계에서 나는지 보인다",
    changed_variable="없음(기준선)",
    controls=["같은 case 선택", "같은 응답 schema", "temperature는 공급자 기본값"],
    comparison=["같은 이미지 기준선: product-extract-and-translate", "규칙 바닥값: empty-text · copy-source"],
    selection_reason="dev 전체(작은 합성 fixture)",
)

if MODE == "provider" and not ALLOW_EXTERNAL:
    raise PermissionError("provider 모드는 ALLOW_EXTERNAL = True(외부 전송 승인)가 필요하다")

In [ ]:
ocr = load_dataset("sample-ocr-boundaries", SPLIT)
translation = load_dataset("sample-translation-boundaries", SPLIT)
OCR_CASES, TR_CASES = sorted(ocr.frame["id"]), sorted(translation.frame["id"])
if MODE == "provider":
    blocked = transfer_blockers(ocr, OCR_CASES) + transfer_blockers(translation, TR_CASES)
    if blocked:
        raise PermissionError(f"외부 전송 태그가 없는 case: {blocked}")
display(ocr.frame[["id", "language", "tokenizer", "source_group", "tags"]])
display(translation.frame[["id", "source_language", "critical_span_count", "review", "source_group"]])

In [ ]:
def caller(spec, prompt_file, smoke_file):
    """MODE에 맞는 답 얻는 방법과 후보 설명."""
    if MODE == "provider":
        candidate = t.Candidate(f"{BASE_ID}-{spec.schema_name.replace('_', '-')}", "fixed-prompt-model", PROVIDER, MODEL, load_prompt(LAB / "prompts" / prompt_file), base_url=BASE_URL, prompt_ref=f"tools/evals/lab/prompts/{prompt_file}")
        return candidate, t.provider_caller(candidate, spec)
    return t.Candidate(f"smoke-{smoke_file.removesuffix('.json')}", "smoke-fixture"), t.fixture_caller(SMOKE / smoke_file)


def run(spec, name, inputs, prompt_file, smoke_file):
    candidate, how = caller(spec, prompt_file, smoke_file)
    run_id = unique_run_id(f"{BASE_ID}-{name}", OUT)
    records = t.run_task(spec, how, inputs, run_id=run_id, max_calls=MAX_CALLS, dry_run=MODE == "dry-run")
    return {"spec": spec, "candidate": candidate, "inputs": inputs, "records": records}


runs = {
    "ocr": run(t.OCR, "ocr", t.image_inputs(ocr, OCR_CASES), "product-extract-text.md", "ocr.json"),
    "translation": run(t.TRANSLATION, "translation", t.text_inputs(translation, TR_CASES), "translate-text-v1.md", "translation.json"),
    "same-image": run(t.EXTRACT_AND_TRANSLATE, "same-image", t.image_inputs(ocr, OCR_CASES), "product-extract-and-translate.md", "extract-and-translate.json"),
}
chain_inputs = t.chain_inputs(runs["ocr"]["records"], ocr, translation)
runs["chain"] = run(t.TRANSLATION, "chain", chain_inputs, "translate-text-v1.md", "translation.json")
for name, r in runs.items():
    print(name, r["records"][0].run_id, {s: sum(x.state == s for x in r["records"]) for s in ("answered", "contract-error", "call-failed", "not-run")})

In [ ]:
# 연쇄 진단은 OCR case id로 기록하고, gold 번역 case는 `gold`에 둔다. smoke 번역 답은 gold case id로만 기록돼 있어 여기서는 not-run이 정상이다.
# 연쇄 진단 답은 gold 원문 번역 점수에 합치지 않는다.
records_frame(runs["chain"]["records"])[["case_id", "input", "state", "problem", "upstream", "gold"]]

In [ ]:
# 규칙 바닥값. 모델을 부르지 않는다 — 모델 성능이 아니라 채점 범위의 바닥이다.
floors = {
    "ocr-floor": ("ocr", t.OCR, t.empty_text_baseline(), t.image_inputs(ocr, OCR_CASES), t.Candidate("rule-empty-text", "rule-baseline")),
    "translation-floor": ("translation", t.TRANSLATION, t.copy_source_baseline(), t.text_inputs(translation, TR_CASES), t.Candidate("rule-copy-source", "rule-baseline")),
}
for name, (_, spec, how, inputs, candidate) in floors.items():
    runs[name] = {"spec": spec, "candidate": candidate, "inputs": inputs, "records": t.run_task(spec, how, inputs, run_id=unique_run_id(f"{BASE_ID}-{name}", OUT))}

In [ ]:
# 답을 gold 옆에 놓고 본다. 점수는 내지 않는다 — 채점은 아래 Go replay가 한다.
import difflib

gold_text = {c["id"]: c["expected"]["textExtraction"]["text"] for c in ocr.cases}
for r in runs["ocr"]["records"]:
    print(f"--- {r.case_id} · {r.state}" + (f" · {r.problem}" if r.problem else ""))
    if r.output is not None:
        print("\n".join(difflib.unified_diff(gold_text[r.case_id].splitlines(), r.output.get("text", "").splitlines(), "gold", "ocr", lineterm="")))

In [ ]:
# 파일로 내보낸다: <runId>.research.jsonl · <runId>.run.json, Go 과제가 있으면 <runId>.jsonl · <runId>.json(replay)
TEMPLATES = {"text-extraction": VARIANTS / "text-extraction-replay.json", "translation": VARIANTS / "translation-replay.json"}
DATASETS = {"text-extraction": ocr, "translation": translation, "extract-and-translate": ocr}
commands = []
for name, r in runs.items():
    spec, records = r["spec"], r["records"]
    paths = t.export_run(records, r["inputs"], spec=spec, candidate=r["candidate"], plan=PLAN, dataset=DATASETS[spec.task], out=OUT,
                         max_calls=MAX_CALLS, dry_run=MODE == "dry-run", template=TEMPLATES.get(spec.replay_task))
    print(name, {k: (v.name if v else None) for k, v in paths.items()})
    if paths["replay"]:
        dataset = DATASETS[spec.task].name
        commands.append((records[0].run_id, f"pnpm eval replay --dataset {dataset} --split {SPLIT} --allow-drafts --variant {paths['variant'].relative_to(ROOT)} --predictions {paths['replay'].relative_to(ROOT)} --run-id {records[0].run_id}"))
print("\n저장소 루트에서 실행(산출물은 git 제외 경로 tools/evals/results):")
for _, command in commands:
    print(command)

In [ ]:
# Go가 채점한 결과 읽기. replay를 돌린 run만 보인다. 값마다 availability(measured · not-applicable · unavailable · unsupported)를 같이 본다.
import json

RESULTS = ROOT / "tools/evals/results"
for run_id, _ in commands:
    summary_path = RESULTS / run_id / "summary.json"
    if not summary_path.exists():
        print(run_id, "아직 replay 안 함")
        continue
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    quality = summary["variants"][0]["quality"][0]
    block = quality.get("text") or quality.get("translation")
    print(run_id, "mode", summary["mode"], "officialEligible", summary["officialEligible"], "status", summary["status"])
    print({k: v for k, v in block.items() if isinstance(v, dict) and "availability" in v})

In [ ]:
# 오류 분석: 상태를 태그(언어 · 레이아웃 · 경계)별로 본다.
frame = records_frame(runs["ocr"]["records"]).merge(ocr.frame[["id", "tags"]], left_on="case_id", right_on="id")
frame = frame.explode("tags")
frame[frame["tags"].str.contains(":", na=False)].pivot_table(index="tags", columns="state", values="case_id", aggfunc="count", fill_value=0)